# Stage 2 - judge (Kaggle GPU T4 x2, Internet ON)
Add your `gen_*.zip` as a private Kaggle Dataset input, set `GEN_ZIP` below. Judge = open model (deviation from GPT-4o-mini; **untested**). Exports only small verdict files (no responses).

In [ ]:
import os, glob, subprocess
GEN_ZIP = glob.glob("/kaggle/input/*/gen_*.zip")[0]; print(GEN_ZIP)
JUDGE_HF = "Qwen/Qwen2.5-7B-Instruct"; JUDGE_ID = "qwen25-7b"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs && unzip -qo {GEN_ZIP} -d runs
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

In [ ]:
for d in sorted(glob.glob("runs/*/seed*")):
    print("judging", d)
    subprocess.run(f"python scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF}", shell=True, check=True)

In [ ]:
# optional faster path (UNTESTED on T4): serve with vLLM, then call 11_run_judge.py with --endpoint http://127.0.0.1:8000/v1/chat/completions --model <name>
# !pip install -q vllm && nohup vllm serve Qwen/Qwen2.5-7B-Instruct --dtype half --tensor-parallel-size 2 --max-model-len 4096 > vllm.log 2>&1 &

In [ ]:
!find runs -name "*.judged_*.jsonl" -o -name "gate_scores_*.csv" | zip -q judged_{JUDGE_ID}.zip -@
!ls -la judged_*.zip; rm -rf /tmp/REINS runs/*/seed*/*.jsonl.bak
# download judged_*.zip; extract into the repo root next to your runs/ folder (merge, do not delete earlier results)